# 07 Kamera-Charakterisierung

**Eingabe**: rohe Bayer-Bilder (FITS, ADU, unkalibriert): Bias, Flat-Paare, Linearitätsreihe, optional Darks  
**Ausgabe**: Kameraprofil — Offset, Maximalwert, Gain, Ausleserauschen, Full Well, Linearitätsbereich, Dunkelstrom **je Farbebene** und je Einstellung → `PHOTOMETRIE_PY_DATA/` und `observer.json`

**Warum**: Gain und Ausleserauschen gehen in das Fehlerbudget ein (NB04), die Sättigungs- und Linearitätsgrenze bestimmt, wie hell Sterne sein dürfen (Handbuch: unter 75 % des Maximalwerts),
und AAVSO verlangt die Kameradaten (Detector Details). Herstellerkurven weichen mitunter ab — selbst messen. Verfahren: AAVSO DSLR Observing Manual v1.4, Anhang A (Gain, Rauschen, Full Well) und Anhang E (Linearität);
Photon-Transfer-Methode nach Janesick (2001). Details: `varphot/camera.py`, KONZEPT.md §11.8.

### Was aufgenommen werden muss (je Gain-Modus und Temperatur gleich wie später bei den Messungen)

| Ordner | Inhalt | Bemerkung |
|---|---|---|
| `bias/` | ≥ 6 Aufnahmen kürzester Belichtung, Deckel zu | Offset und Ausleserauschen |
| `flats/` | **Paare** gleicher Belichtung bei 8–12 Pegeln, ca. 5–80 % des Maximalwerts | stabile, gleichmäßige Lichtquelle (Leuchtfläche, Flatfield-Panel); nichts gesättigt |
| `linearity/` | Belichtungsreihe 2, 4, 6 … s, **je dreimal**, bis einige Stufen über die Sättigung | gleiche Lichtquelle, gleicher Ausschnitt; so entsteht auch der Maximalwert |
| `darks/` (optional) | ≥ 5 Belichtungszeiten, je 2 Bilder, Temperatur stabil | Dunkelstrom (bei sehr kleinem Dunkelstrom nur Obergrenze) |

Messung immer **je einzelner Farbebene** (R, G1, G2, B) im Bildmittelpunkt — nie am gemittelten Grün (verkleinert die Streuung und verfälscht Gain/Rauschen).
Belichtungen von weniger als ca. 10 s können durch die reale Verschlusszeit abweichen (Ares-C Pro: Rolling Shutter).

## Schritt 1 — Einstellungen

`MODE="demo"`: synthetische Bilder mit **bekannter Wahrheit** (zeigt, dass die Auswertung sie wiederfindet; es wird nichts gespeichert).
`MODE="real"`: eigene Bilder aus `DIRS`. Umschalten per Umgebungsvariable `VARPHOT_NB07_MODE=real` oder hier.

In [ ]:
import os, json, datetime, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from varphot import (pair_stats, gain_from_pair, ptc_fit, read_noise_e, full_well_e, linearity_curve, dark_current_e_per_s)
from varphot.camera import PLANES, summarize_frame, group_by_exposure
from varphot.observer import Camera, load_config, save_config, data_dir
from varphot.batch import read_frame
from varphot.synthetic import bayer_frame
from varphot.testdata import testdata_dir

MODE    = os.environ.get("VARPHOT_NB07_MODE", "demo")          # "demo" oder "real"
MODEL   = "Player One Ares-C Pro"
SETTING = "Gain 0, LRN, -10 C"                                  # Gain/Modus/Temperatur, auf die sich die Werte beziehen
PATTERN, BITS, ROI = "RGGB", 14, 100                            # Bayer-Muster (Handbuch/Hersteller), ADC-Bittiefe, Mittelbereich je Ebene [px]
BASE = Path(os.environ.get("VARPHOT_CAMERA_DIR") or testdata_dir() / "camera" / "ares-c-pro-gain0-lrn")
DIRS = {k: BASE / k for k in ("bias", "flats", "linearity", "darks")}

# Herstellerwerte zum Vergleich (Gain 0, LRN; AAVSO-Forum „Player One IMX533 settings“ nach den Herstellerdiagrammen; Handbuch V1.0 Dez. 2024)
VENDOR = dict(gain=4.449, read_noise=4.46, full_well=72889)

if MODE == "demo":
    TRUTH = dict(gain=4.4, read_noise=4.5, offset=100.0, max_adu=2 ** BITS - 1, dark=0.5)   # was die Auswertung wiederfinden soll
    print("DEMO-Modus: synthetische Bilder, Wahrheit =", TRUTH)
else:
    print("REAL-Modus: Bilder aus", BASE)

## Schritt 2 — Bilder laden und verdichten

Je Bild bleiben nur der Mittelbereich jeder Farbebene, der Maximalwert und der Median übrig (spart Speicher bei großen Bildern).

In [ ]:
def load_dir(folder):
    recs = []
    for p in sorted(Path(folder).glob("*.fit*")):
        img, hdr, _ = read_frame(p)
        r = summarize_frame(img, PATTERN, ROI)
        r["exptime"], r["file"] = float(hdr.get("EXPTIME", hdr.get("EXPOSURE", np.nan))), p.name
        recs.append(r)
    return recs

def synth(signal_e=0.0, exptime=0.001, dark_e=0.0):
    img = bayer_frame((400, 600), signal_e, TRUTH["gain"], TRUTH["read_noise"], TRUTH["offset"], TRUTH["max_adu"], PATTERN,
                      dark_e=dark_e, rng=RNG)
    r = summarize_frame(img, PATTERN, ROI); r["exptime"] = exptime; r["file"] = "synthetisch"
    return r

if MODE == "demo":
    RNG = np.random.default_rng(7)
    BIAS = [synth() for _ in range(6)]
    levels = [1500, 4000, 9000, 18000, 30000, 42000, 54000, 64000]                     # Elektronen je Pixel
    FLATS = [synth(s, exptime=s / 1000.0) for s in levels for _ in range(2)]
    LIN = [synth(2300.0 * t, exptime=t) for t in range(2, 42, 2) for _ in range(3)]    # sättigt bei ~31 s
    DARKS = [synth(0.0, exptime=t, dark_e=TRUTH["dark"] * t) for t in (10, 30, 60, 120, 240) for _ in range(2)]
else:
    BIAS, FLATS, LIN = (load_dir(DIRS[k]) for k in ("bias", "flats", "linearity"))
    DARKS = load_dir(DIRS["darks"]) if DIRS["darks"].is_dir() else []
print(f"Bias {len(BIAS)}, Flats {len(FLATS)}, Linearität {len(LIN)}, Darks {len(DARKS)} Bilder")

## Schritt 3 — Offset und Maximalwert

**Offset** = Median der Bias-Pegel (Bias-Bild kürzester Belichtung). **Maximalwert** = hellster Pixel der gesättigten Aufnahmen (Linearitätsreihe/Flats); er sollte in allen Ebenen gleich sein und
unter dem nominellen `2^Bits − 1` liegen oder gleich. Wird nichts gesättigt, ist der Wert nur eine Untergrenze — dann `MAX_ADU_NOMINAL` nehmen und kenntlich machen.

In [ ]:
from varphot.camera import _clipped_mean
offset = {p: float(np.median([_clipped_mean(r["rois"][p]) for r in BIAS])) for p in PLANES}
max_adu_meas = float(max(r["max"] for r in LIN + FLATS))
MAX_ADU_NOMINAL = 2 ** BITS - 1
saturated = max_adu_meas >= 0.98 * MAX_ADU_NOMINAL
MAX_ADU = max_adu_meas if saturated else MAX_ADU_NOMINAL
print("Offset je Ebene [ADU]:", {k: round(v, 1) for k, v in offset.items()})
print(f"Maximalwert gemessen {max_adu_meas:.0f} ADU (nominell {MAX_ADU_NOMINAL}) —", "gesättigt, gemessener Wert gilt" if saturated else "NICHT gesättigt, nominell verwendet")

## Schritt 4 — Gain (Photon-Transfer) je Farbebene

Je Flat-Paar: Mittelwert und zeitliche Varianz (Differenz der zwei Bilder entfernt Flatfield-Struktur). Gerade `Varianz = Signal/Gain + const` über alle Pegel; Gain = 1/Steigung.
Gegenprobe: Einzelpaar-Formel des Handbuchs (Anhang A) beim Paar nahe 2/3 des Maximalwerts. Die Ebenen sollten denselben Gain liefern.

In [ ]:
pairs = group_by_exposure(FLATS)
rows, ptc_data = [], {}
for p in PLANES:
    means, varis = [], []
    for t, grp in pairs.items():
        m, v = pair_stats(grp[0]["rois"][p], grp[1]["rois"][p], ROI)
        means.append(m); varis.append(v)
    fit = ptc_fit(means, varis, offset[p], MAX_ADU)
    # Einzelpaar nahe 2/3 des Maximalwerts + Bias-Paar
    k = int(np.argmin([abs(m - offset[p] - 0.66 * (MAX_ADU - offset[p])) for m in means]))
    grp = list(pairs.values())[k]
    g_pair = gain_from_pair(grp[0]["rois"][p], grp[1]["rois"][p], BIAS[0]["rois"][p], BIAS[1]["rois"][p], ROI)
    rn = read_noise_e(BIAS[0]["rois"][p], BIAS[1]["rois"][p], fit["gain"], ROI)
    rows.append(dict(Ebene=p, Offset_ADU=offset[p], Gain_PTC=fit["gain"], Gain_Paar=g_pair, R2=fit["r2"], n_Paare=fit["n"],
                     Ausleserauschen_e=rn, FullWell_e=full_well_e(fit["gain"], MAX_ADU)))
    ptc_data[p] = (np.array(means) - offset[p], np.array(varis), fit)
TAB = pd.DataFrame(rows).set_index("Ebene")
display(TAB.round(3))
spread = (TAB["Gain_PTC"].max() - TAB["Gain_PTC"].min()) / TAB["Gain_PTC"].mean() * 100
print(f"Streuung des Gains über die Ebenen: {spread:.1f} % (sollte klein sein; > 5 % → Pegel/Lichtquelle/Muster prüfen)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for p, (s, v, fit) in ptc_data.items():
    ax[0].plot(s, v, "o", ms=4, label=p)
    xs = np.linspace(0, s.max(), 20); ax[0].plot(xs, fit["slope"] * xs + fit["intercept"], lw=0.8)
ax[0].set_xlabel("Signal − Offset [ADU]"); ax[0].set_ylabel("Varianz [ADU²]"); ax[0].set_title("Photon-Transfer-Kurve (Steigung = 1/Gain)"); ax[0].legend()
ax[1].bar(TAB.index, TAB["Gain_PTC"], color="C0", label="PTC"); ax[1].plot(TAB.index, TAB["Gain_Paar"], "ko", label="Einzelpaar")
ax[1].set_ylabel("Gain [e⁻/ADU]"); ax[1].set_title("Gain je Ebene"); ax[1].legend()
plt.tight_layout(); plt.show()

## Schritt 5 — Linearität

Mittelwert eines festen Bereichs (Grünebene G1) gegen die Belichtungszeit; gemittelt über die drei Bilder je Belichtung. Zählrate `(ADU − a)/t` gegen eine Referenz-Gerade; **linear** heißt hier |Abweichung| ≤ 1 %
(Startwert). Das ergibt die ADU-Grenzen, innerhalb derer Sterne liegen dürfen. Auffällige Abweichungen bei kurzen Zeiten → reale Verschlusszeit/Rolling Shutter.

In [ ]:
by_t = {}
for r in LIN:
    by_t.setdefault(round(r["exptime"], 3), []).append(r)
ts = np.array(sorted(by_t))
mean_adu = np.array([np.mean([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]) for t in ts])
spread_t = np.array([np.std([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]) / max(np.mean([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]), 1) * 100 for t in ts])
if spread_t.max() > 0.5:
    print(f"WARNUNG: Lichtquelle instabil? Streuung der drei Bilder bis {spread_t.max():.2f} % — Linearitätstest dann nicht gültig (Handbuch Anhang E)")
LINRES = linearity_curve(ts, mean_adu, max_adu=MAX_ADU, tol_pct=1.0)
print(f"Offset (Achsenabschnitt) {LINRES['a']:.1f} ADU, Zählrate {LINRES['b']:.1f} ADU/s")
print(f"linear (±1 %) von {LINRES['t_min']:.0f} bis {LINRES['t_max']:.0f} s = {LINRES['adu_min']:.0f} bis {LINRES['adu_max']:.0f} ADU "
      f"({100 * LINRES['adu_max'] / MAX_ADU:.0f} % des Maximalwerts)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ts, mean_adu, "o-", ms=4); ax[0].axhline(MAX_ADU, color="r", ls=":", label="Maximalwert"); ax[0].set_xlabel("Belichtung [s]"); ax[0].set_ylabel("Mittelwert G1 [ADU]"); ax[0].legend()
ax[1].plot(ts, LINRES["residual_pct"], "o-", ms=4); ax[1].axhspan(-1, 1, color="g", alpha=0.15); ax[1].set_ylim(-10, 10)
ax[1].set_xlabel("Belichtung [s]"); ax[1].set_ylabel("Zählratenabweichung [%]"); ax[1].set_title("grün = linear (±1 %)")
plt.tight_layout(); plt.show()

## Schritt 6 — Dunkelstrom (optional)

Steigung der mittleren Dunkelwerte gegen die Belichtungszeit mal Gain. Gilt nur bei stabil geregelter Temperatur. Der Hersteller nennt 4,4·10⁻⁵ e⁻/s/Pixel bei −20 °C — das liegt unter der Nachweisgrenze
dieser Methode; dann ist nur eine **Obergrenze** aussagekräftig.

In [ ]:
GAIN = float(TAB["Gain_PTC"].mean())
if DARKS:
    d = {}
    for r in DARKS:
        d.setdefault(round(r["exptime"], 3), []).append(np.mean([_clipped_mean(r["rois"][p]) for p in PLANES]))
    dts = np.array(sorted(d)); dm = np.array([np.mean(d[t]) for t in dts])
    DARK = dark_current_e_per_s(dts, dm, GAIN)
    print(f"Dunkelstrom ≈ {DARK:.4f} e⁻/s/Pixel (bei der Temperatur der Aufnahmen; Hersteller: 0,000044 bei −20 °C, 0,000996 bei 0 °C)")
else:
    DARK = None
    print("keine Darks — Dunkelstrom nicht gemessen")

## Schritt 7 — Ergebnis, Vergleich mit dem Hersteller, Speichern

Abweichungen von mehr als 10 % zum Herstellerwert werden markiert. **Gespeichert wird nur im Modus `real`** (Kameraprofil in `PHOTOMETRIE_PY_DATA/` und der Abschnitt `camera` in `observer.json`).

In [ ]:
RES = dict(gain_e_per_adu=GAIN, readout_noise_e=float(TAB["Ausleserauschen_e"].median()), full_well_e=full_well_e(GAIN, MAX_ADU),
           offset_adu=float(np.mean(list(offset.values()))), saturation_adu=MAX_ADU)
cmp = pd.DataFrame([
    dict(Größe="Gain [e⁻/ADU]", gemessen=RES["gain_e_per_adu"], Hersteller=VENDOR["gain"]),
    dict(Größe="Ausleserauschen [e⁻]", gemessen=RES["readout_noise_e"], Hersteller=VENDOR["read_noise"]),
    dict(Größe="Full Well [e⁻]", gemessen=RES["full_well_e"], Hersteller=VENDOR["full_well"]),
]).set_index("Größe")
cmp["Abweichung_%"] = 100 * (cmp["gemessen"] / cmp["Hersteller"] - 1)
cmp["Urteil"] = np.where(cmp["Abweichung_%"].abs() > 10, "⚠ > 10 %", "ok")
display(cmp.round(2))
if MODE == "demo":
    print("\nWahrheit der Demo:", {k: TRUTH[k] for k in ("gain", "read_noise")},
          "→ gefunden: Gain %.2f, Ausleserauschen %.2f" % (RES["gain_e_per_adu"], RES["readout_noise_e"]))

In [ ]:
lin_text = (f"linear ±1 % von {LINRES['adu_min']:.0f} bis {LINRES['adu_max']:.0f} ADU ({SETTING})")
CAM = Camera(model=MODEL, cam_type="CMOS", gain_e_per_adu=round(RES["gain_e_per_adu"], 3), readout_noise_e=round(RES["readout_noise_e"], 2),
             dark_current_e_per_s=None if DARK is None else round(DARK, 5), dark_temp_c=None, linearity=lin_text, bayer_pattern=PATTERN,
             bit_depth=BITS, saturation_adu=round(RES["saturation_adu"], 1), offset_adu=round(RES["offset_adu"], 1),
             full_well_e=round(RES["full_well_e"]), gain_setting=SETTING,
             source=f"gemessen (NB07, {datetime.date.today().isoformat()})")
if MODE == "real":
    obs, own_sites = load_config()
    obs.camera = CAM
    path = save_config(obs, own_sites)
    prof = data_dir() / ("camera_" + re.sub(r"[^A-Za-z0-9]+", "_", f"{MODEL}_{SETTING}").strip("_") + ".json")
    prof.write_text(json.dumps({"camera": CAM.__dict__, "je_ebene": TAB.round(4).to_dict(orient="index"),
                                "linearitaet": {k: LINRES[k] for k in ("a", "b", "t_min", "t_max", "adu_min", "adu_max")}},
                               indent=2, ensure_ascii=False, default=float) + "\n", encoding="utf-8")
    print("gespeichert:", path, "und", prof)
else:
    print("DEMO: nichts gespeichert. Profil wäre:", CAM)